In [1]:
import pandas as pd

semua = pd.read_csv(
    r"C:\Users\s2\Documents\FIXRKI\FIXbanget\newscema\Full_Client\test_per_client_per_seed.csv"
)

metrik = ["acc", "prec", "rec", "f1", "auc", "prauc"]

rata_seed = (
    semua.groupby(["algoritma", "seed"])[metrik]
    .mean()
    .round(4)
    .reset_index()
)

print(rata_seed)

                 algoritma  seed     acc    prec     rec      f1     auc  \
0                   fedavg     0  0.7064  0.3726  0.6165  0.4633  0.7379   
1                   fedavg    42  0.7037  0.3702  0.6206  0.4627  0.7357   
2                   fedavg   123  0.6993  0.3688  0.6394  0.4666  0.7386   
3                   fedavg  1337  0.7005  0.3679  0.6258  0.4623  0.7391   
4                   fedavg  3407  0.6980  0.3662  0.6328  0.4629  0.7361   
5     fedpairwise_T0.01_d0     0  0.7082  0.3753  0.6209  0.4664  0.7398   
6     fedpairwise_T0.01_d0    42  0.7037  0.3701  0.6194  0.4620  0.7399   
7     fedpairwise_T0.01_d0   123  0.6988  0.3667  0.6312  0.4626  0.7397   
8     fedpairwise_T0.01_d0  1337  0.7086  0.3756  0.6183  0.4662  0.7404   
9     fedpairwise_T0.01_d0  3407  0.7000  0.3685  0.6337  0.4648  0.7394   
10  fedpairwise_T0.01_d0.3     0  0.7088  0.3762  0.6232  0.4679  0.7404   
11  fedpairwise_T0.01_d0.3    42  0.7033  0.3704  0.6230  0.4633  0.7400   
12  fedpairw

In [2]:
import numpy as np
import pandas as pd
from scipy import stats

# --- data: hasil rata-rata per (algoritma, seed) ---
df = pd.read_csv(
     r"C:\Users\s2\Documents\FIXRKI\FIXbanget\newscema\Full_Client\test_per_client_per_seed.csv"
)

metrik   = ["acc", "prec", "rec", "f1", "auc", "prauc"]
baseline = "fedavg"
varian   = ["fedpairwise_T0.01_d0",
            "fedpairwise_T0.01_d0.3",
            "fedpairwise_T0.01_d0.5"]
ALPHA = 0.05

base = df[df.algoritma == baseline].set_index("seed")

rows = []
for v in varian:
    cur = df[df.algoritma == v].set_index("seed")
    seeds = cur.index.intersection(base.index).sort_values()   # pasangkan per seed

    for m in metrik:
        x = cur.loc[seeds, m].astype(float)
        y = base.loc[seeds, m].astype(float)
        d = (x - y).dropna()
        n = len(d)

        mean_d = d.mean()
        sd_d   = d.std(ddof=1) if n > 1 else np.nan

        if n >= 2 and sd_d > 0:
            t, p = stats.ttest_rel(x.loc[d.index], y.loc[d.index])  # dua sisi
            se   = sd_d / np.sqrt(n)
            tcr  = stats.t.ppf(1 - ALPHA / 2, n - 1)
            ci_lo, ci_hi = mean_d - tcr * se, mean_d + tcr * se
            dz   = mean_d / sd_d                                    # Cohen's dz
        else:
            t = p = ci_lo = ci_hi = dz = np.nan

        rows.append({
            "pembanding": f"{v} vs {baseline}",
            "metrik": m,
            "n": n,
            "mean_variant": x.mean(),
            "mean_fedavg": y.mean(),
            "selisih_mean": mean_d,
            "std_selisih": sd_d,
            "CI95_low": ci_lo,
            "CI95_high": ci_hi,
            "t": t,
            "df": n - 1,
            "p_value": p,
            "cohen_dz": dz,
            "menang": f"{int((d > 0).sum())}/{n}",
        })

hasil = pd.DataFrame(rows)

# --- koreksi Holm (untuk 18 uji sekaligus) ---
def holm(pvals):
    p = np.asarray(pvals, dtype=float)
    adj = np.full_like(p, np.nan)
    ok = ~np.isnan(p)
    idx = np.argsort(p[ok])
    pv = p[ok][idx]
    m = len(pv)
    h = np.maximum.accumulate((m - np.arange(m)) * pv)
    h = np.minimum(h, 1.0)
    tmp = np.empty(m)
    tmp[idx] = h
    adj[ok] = tmp
    return adj

hasil["p_holm"] = holm(hasil["p_value"])
hasil["signifikan (p<0.05)"] = np.where(hasil["p_value"] < ALPHA, "Ya", "Tidak")
hasil["signifikan (Holm)"]   = np.where(hasil["p_holm"] < ALPHA, "Ya", "Tidak")
hasil["arah"] = np.where(hasil["selisih_mean"] > 0, "variant > fedavg",
                         np.where(hasil["selisih_mean"] < 0, "variant < fedavg", "sama"))

pd.set_option("display.width", 250, "display.max_columns", None)
print(hasil.round(4).to_string(index=False))

# hasil.round(6).to_csv(
#     r"C:\Users\s2\Documents\FIXRKI\FIXbanget\newscema\Top20\uji_t_berpasangan.csv",
#     index=False,
# )

                      pembanding metrik   n  mean_variant  mean_fedavg  selisih_mean  std_selisih  CI95_low  CI95_high       t  df  p_value  cohen_dz  menang  p_holm signifikan (p<0.05) signifikan (Holm)             arah
  fedpairwise_T0.01_d0 vs fedavg    acc 220        0.7039       0.7016        0.0023       0.0120    0.0007     0.0039 18.7076 219   0.0000    0.1897 123/220  0.0000                  Ya                Ya variant > fedavg
  fedpairwise_T0.01_d0 vs fedavg   prec 220        0.3712       0.3692        0.0021       0.0154    0.0000     0.0041 13.2089 219   0.0000    0.1340 126/220  0.0000                  Ya                Ya variant > fedavg
  fedpairwise_T0.01_d0 vs fedavg    rec 220        0.6247       0.6270       -0.0023       0.0284   -0.0061     0.0015 -8.0408 219   0.0000   -0.0815  90/220  0.0000                  Ya                Ya variant < fedavg
  fedpairwise_T0.01_d0 vs fedavg     f1 220        0.4644       0.4635        0.0008       0.0181   -0.0016     0.00

In [3]:
hasil

,pembanding,metrik,n,mean_variant,mean_fedavg,selisih_mean,std_selisih,CI95_low,CI95_high,t,df,p_value,cohen_dz,menang,p_holm,signifikan (p<0.05),signifikan (Holm),arah
0,fedpairwise_T0.01_d0 vs fedavg,acc,220,0.703870,0.701596,0.002274,0.011984,0.000681,0.003866,18.707588,219,9.598826e-77,0.189720,123/220,7.679061e-76,Ya,Ya,variant > fedavg
1,fedpairwise_T0.01_d0 vs fedavg,prec,220,0.371214,0.369152,0.002062,0.015396,0.000017,0.004108,13.208911,219,1.710321e-39,0.133956,126/220,8.551607e-39,Ya,Ya,variant > fedavg
2,fedpairwise_T0.01_d0 vs fedavg,rec,220,0.624716,0.627032,-0.002316,0.028402,-0.006090,0.001458,-8.040805,219,9.970745e-16,-0.081545,90/220,2.991223e-15,Ya,Ya,variant < fedavg
3,fedpairwise_T0.01_d0 vs fedavg,f1,220,0.464396,0.463550,0.000846,0.018126,-0.001562,0.003255,4.604357,219,4.190167e-06,0.046694,117/220,8.380334e-06,Ya,Ya,variant > fedavg
4,fedpairwise_T0.01_d0 vs fedavg,auc,220,0.739817,0.737469,0.002348,0.007486,0.001354,0.003343,30.935258,219,1.846609e-200,0.313725,142/220,2.400592e-199,Ya,Ya,variant > fedavg
5,fedpairwise_T0.01_d0 vs fedavg,prauc,220,0.439828,0.435160,0.004668,0.014296,0.002768,0.006567,32.193890,219,4.052784e-216,0.326489,149/220,5.673898e-215,Ya,Ya,variant > fedavg
6,fedpairwise_T0.01_d0.3 vs fedavg,acc,220,0.703572,0.701596,0.001975,0.010220,0.000617,0.003333,19.059605,219,1.549302e-79,0.193290,123/220,1.394372e-78,Ya,Ya,variant > fedavg
7,fedpairwise_T0.01_d0.3 vs fedavg,prec,220,0.371446,0.369152,0.002294,0.012983,0.000569,0.004019,17.420383,219,6.032482e-67,0.176666,128/220,3.619489e-66,Ya,Ya,variant > fedavg
8,fedpairwise_T0.01_d0.3 vs fedavg,rec,220,0.627942,0.627032,0.000910,0.025066,-0.002421,0.004240,3.577795,219,3.481973e-04,0.036284,103/220,3.481973e-04,Ya,Ya,variant > fedavg
9,fedpairwise_T0.01_d0.3 vs fedavg,f1,220,0.465513,0.463550,0.001964,0.015431,-0.000086,0.004014,12.550002,219,7.553059e-36,0.127274,129/220,3.021224e-35,Ya,Ya,variant > fedavg


In [4]:
import numpy as np
import pandas as pd
from scipy import stats

# --- data: hasil rata-rata per (algoritma, seed) ---
df = pd.read_csv(
     r"C:\Users\s2\Documents\FIXRKI\FIXbanget\newscema\Full_Client\test_per_client_per_seed.csv"
)

metrik   = ["acc", "prec", "rec", "f1", "auc", "prauc"]
baseline = "fedprox"
varian   = ["fedpairwise_T0.01_d0",
            "fedpairwise_T0.01_d0.3",
            "fedpairwise_T0.01_d0.5"]
ALPHA = 0.05

base = df[df.algoritma == baseline].set_index("seed")

rows = []
for v in varian:
    cur = df[df.algoritma == v].set_index("seed")
    seeds = cur.index.intersection(base.index).sort_values()   # pasangkan per seed

    for m in metrik:
        x = cur.loc[seeds, m].astype(float)
        y = base.loc[seeds, m].astype(float)
        d = (x - y).dropna()
        n = len(d)

        mean_d = d.mean()
        sd_d   = d.std(ddof=1) if n > 1 else np.nan

        if n >= 2 and sd_d > 0:
            t, p = stats.ttest_rel(x.loc[d.index], y.loc[d.index])  # dua sisi
            se   = sd_d / np.sqrt(n)
            tcr  = stats.t.ppf(1 - ALPHA / 2, n - 1)
            ci_lo, ci_hi = mean_d - tcr * se, mean_d + tcr * se
            dz   = mean_d / sd_d                                    # Cohen's dz
        else:
            t = p = ci_lo = ci_hi = dz = np.nan

        rows.append({
            "pembanding": f"{v} vs {baseline}",
            "metrik": m,
            "n": n,
            "mean_variant": x.mean(),
            "mean_fedavg": y.mean(),
            "selisih_mean": mean_d,
            "std_selisih": sd_d,
            "CI95_low": ci_lo,
            "CI95_high": ci_hi,
            "t": t,
            "df": n - 1,
            "p_value": p,
            "cohen_dz": dz,
            "menang": f"{int((d > 0).sum())}/{n}",
        })

hasil2 = pd.DataFrame(rows)

# --- koreksi Holm (untuk 18 uji sekaligus) ---
def holm(pvals):
    p = np.asarray(pvals, dtype=float)
    adj = np.full_like(p, np.nan)
    ok = ~np.isnan(p)
    idx = np.argsort(p[ok])
    pv = p[ok][idx]
    m = len(pv)
    h = np.maximum.accumulate((m - np.arange(m)) * pv)
    h = np.minimum(h, 1.0)
    tmp = np.empty(m)
    tmp[idx] = h
    adj[ok] = tmp
    return adj

hasil2["p_holm"] = holm(hasil2["p_value"])
hasil2["signifikan (p<0.05)"] = np.where(hasil2["p_value"] < ALPHA, "Ya", "Tidak")
hasil2["signifikan (Holm)"]   = np.where(hasil2["p_holm"] < ALPHA, "Ya", "Tidak")
hasil2["arah"] = np.where(hasil2["selisih_mean"] > 0, "variant > fedavg",
                         np.where(hasil2["selisih_mean"] < 0, "variant < fedavg", "sama"))

pd.set_option("display.width", 250, "display.max_columns", None)
print(hasil2.round(4).to_string(index=False))

# hasil.round(6).to_csv(
#     r"C:\Users\s2\Documents\FIXRKI\FIXbanget\newscema\Top20\uji_t_berpasangan.csv",
#     index=False,
# )

                       pembanding metrik   n  mean_variant  mean_fedavg  selisih_mean  std_selisih  CI95_low  CI95_high        t  df  p_value  cohen_dz  menang  p_holm signifikan (p<0.05) signifikan (Holm)             arah
  fedpairwise_T0.01_d0 vs fedprox    acc 220        0.7039       0.7001        0.0038       0.0117    0.0022     0.0053  31.6346 219   0.0000    0.3208 133/220  0.0000                  Ya                Ya variant > fedavg
  fedpairwise_T0.01_d0 vs fedprox   prec 220        0.3712       0.3684        0.0029       0.0153    0.0008     0.0049  18.3872 219   0.0000    0.1865 131/220  0.0000                  Ya                Ya variant > fedavg
  fedpairwise_T0.01_d0 vs fedprox    rec 220        0.6247       0.6325       -0.0078       0.0286   -0.0116    -0.0040 -26.8346 219   0.0000   -0.2721  70/220  0.0000                  Ya                Ya variant < fedavg
  fedpairwise_T0.01_d0 vs fedprox     f1 220        0.4644       0.4644        0.0000       0.0183   -0.0024

In [5]:
hasil2

,pembanding,metrik,n,mean_variant,mean_fedavg,selisih_mean,std_selisih,CI95_low,CI95_high,t,df,p_value,cohen_dz,menang,p_holm,signifikan (p<0.05),signifikan (Holm),arah
0,fedpairwise_T0.01_d0 vs fedprox,acc,220,0.703870,0.700116,0.003754,0.011702,0.002199,0.005309,31.634641,219,4.244996e-209,0.320818,133/220,5.518495e-208,Ya,Ya,variant > fedavg
1,fedpairwise_T0.01_d0 vs fedprox,prec,220,0.371214,0.368359,0.002855,0.015313,0.000821,0.004890,18.387210,219,3.030415e-74,0.186471,131/220,1.818249e-73,Ya,Ya,variant > fedavg
2,fedpairwise_T0.01_d0 vs fedprox,rec,220,0.624716,0.632508,-0.007792,0.028632,-0.011596,-0.003987,-26.834642,219,4.616565e-153,-0.272139,70/220,4.616565e-152,Ya,Ya,variant < fedavg
3,fedpairwise_T0.01_d0 vs fedprox,f1,220,0.464396,0.464384,0.000012,0.018262,-0.002414,0.002439,0.066041,219,9.473462e-01,0.000670,111/220,9.473462e-01,Tidak,Tidak,variant > fedavg
4,fedpairwise_T0.01_d0 vs fedprox,auc,220,0.739817,0.738456,0.001362,0.007138,0.000413,0.002310,18.814574,219,1.376344e-77,0.190805,140/220,9.634410e-77,Ya,Ya,variant > fedavg
5,fedpairwise_T0.01_d0 vs fedprox,prauc,220,0.439828,0.436741,0.003087,0.013761,0.001258,0.004915,22.118501,219,8.513732e-106,0.224311,144/220,7.662359e-105,Ya,Ya,variant > fedavg
6,fedpairwise_T0.01_d0.3 vs fedprox,acc,220,0.703572,0.700116,0.003456,0.010542,0.002055,0.004857,32.327689,219,8.194541e-218,0.327846,134/220,1.147236e-216,Ya,Ya,variant > fedavg
7,fedpairwise_T0.01_d0.3 vs fedprox,prec,220,0.371446,0.368359,0.003087,0.014015,0.001225,0.004949,21.718345,219,3.686452e-102,0.220253,133/220,2.949161e-101,Ya,Ya,variant > fedavg
8,fedpairwise_T0.01_d0.3 vs fedprox,rec,220,0.627942,0.632508,-0.004566,0.025908,-0.008009,-0.001124,-17.379641,219,1.202315e-66,-0.176253,81/220,6.011576e-66,Ya,Ya,variant < fedavg
9,fedpairwise_T0.01_d0.3 vs fedprox,f1,220,0.465513,0.464384,0.001130,0.016642,-0.001082,0.003341,6.693974,219,2.292376e-11,0.067886,115/220,4.584752e-11,Ya,Ya,variant > fedavg


In [6]:
import numpy as np
import pandas as pd
from scipy import stats

# --- data: hasil rata-rata per (algoritma, seed) ---
df = pd.read_csv(
     r"C:\Users\s2\Documents\FIXRKI\FIXbanget\newscema\Full_Client\test_per_client_per_seed.csv"
)

metrik   = ["acc", "prec", "rec", "f1", "auc", "prauc"]
baseline = "fedsim_g3"
varian   = ["fedpairwise_T0.01_d0",
            "fedpairwise_T0.01_d0.3",
            "fedpairwise_T0.01_d0.5"]
ALPHA = 0.05

base = df[df.algoritma == baseline].set_index("seed")

rows = []
for v in varian:
    cur = df[df.algoritma == v].set_index("seed")
    seeds = cur.index.intersection(base.index).sort_values()   # pasangkan per seed

    for m in metrik:
        x = cur.loc[seeds, m].astype(float)
        y = base.loc[seeds, m].astype(float)
        d = (x - y).dropna()
        n = len(d)

        mean_d = d.mean()
        sd_d   = d.std(ddof=1) if n > 1 else np.nan

        if n >= 2 and sd_d > 0:
            t, p = stats.ttest_rel(x.loc[d.index], y.loc[d.index])  # dua sisi
            se   = sd_d / np.sqrt(n)
            tcr  = stats.t.ppf(1 - ALPHA / 2, n - 1)
            ci_lo, ci_hi = mean_d - tcr * se, mean_d + tcr * se
            dz   = mean_d / sd_d                                    # Cohen's dz
        else:
            t = p = ci_lo = ci_hi = dz = np.nan

        rows.append({
            "pembanding": f"{v} vs {baseline}",
            "metrik": m,
            "n": n,
            "mean_variant": x.mean(),
            "mean_fedavg": y.mean(),
            "selisih_mean": mean_d,
            "std_selisih": sd_d,
            "CI95_low": ci_lo,
            "CI95_high": ci_hi,
            "t": t,
            "df": n - 1,
            "p_value": p,
            "cohen_dz": dz,
            "menang": f"{int((d > 0).sum())}/{n}",
        })

hasil3 = pd.DataFrame(rows)

# --- koreksi Holm (untuk 18 uji sekaligus) ---
def holm(pvals):
    p = np.asarray(pvals, dtype=float)
    adj = np.full_like(p, np.nan)
    ok = ~np.isnan(p)
    idx = np.argsort(p[ok])
    pv = p[ok][idx]
    m = len(pv)
    h = np.maximum.accumulate((m - np.arange(m)) * pv)
    h = np.minimum(h, 1.0)
    tmp = np.empty(m)
    tmp[idx] = h
    adj[ok] = tmp
    return adj

hasil3["p_holm"] = holm(hasil3["p_value"])
hasil3["signifikan (p<0.05)"] = np.where(hasil3["p_value"] < ALPHA, "Ya", "Tidak")
hasil3["signifikan (Holm)"]   = np.where(hasil3["p_holm"] < ALPHA, "Ya", "Tidak")
hasil3["arah"] = np.where(hasil3["selisih_mean"] > 0, "variant > fedavg",
                         np.where(hasil3["selisih_mean"] < 0, "variant < fedavg", "sama"))

pd.set_option("display.width", 250, "display.max_columns", None)
print(hasil3.round(4).to_string(index=False))

# hasil.round(6).to_csv(
#     r"C:\Users\s2\Documents\FIXRKI\FIXbanget\newscema\Top20\uji_t_berpasangan.csv",
#     index=False,
# )

                         pembanding metrik   n  mean_variant  mean_fedavg  selisih_mean  std_selisih  CI95_low  CI95_high        t  df  p_value  cohen_dz  menang  p_holm signifikan (p<0.05) signifikan (Holm)             arah
  fedpairwise_T0.01_d0 vs fedsim_g3    acc 220        0.7039       0.6938        0.0100       0.0152    0.0080     0.0121  65.1364 219      0.0    0.6606 159/220     0.0                  Ya                Ya variant > fedavg
  fedpairwise_T0.01_d0 vs fedsim_g3   prec 220        0.3712       0.3557        0.0155       0.0191    0.0130     0.0181  80.1686 219      0.0    0.8130 177/220     0.0                  Ya                Ya variant > fedavg
  fedpairwise_T0.01_d0 vs fedsim_g3    rec 220        0.6247       0.5947        0.0300       0.0473    0.0238     0.0363  62.6689 219      0.0    0.6355 167/220     0.0                  Ya                Ya variant > fedavg
  fedpairwise_T0.01_d0 vs fedsim_g3     f1 220        0.4644       0.4437        0.0207       0.0252

In [7]:
hasil3

,pembanding,metrik,n,mean_variant,mean_fedavg,selisih_mean,std_selisih,CI95_low,CI95_high,t,df,p_value,cohen_dz,menang,p_holm,signifikan (p<0.05),signifikan (Holm),arah
0,fedpairwise_T0.01_d0 vs fedsim_g3,acc,220,0.703870,0.693837,0.010033,0.015188,0.008015,0.012051,65.136411,219,0.0,0.660571,159/220,0.0,Ya,Ya,variant > fedavg
1,fedpairwise_T0.01_d0 vs fedsim_g3,prec,220,0.371214,0.355665,0.015550,0.019126,0.013008,0.018091,80.168577,219,0.0,0.813017,177/220,0.0,Ya,Ya,variant > fedavg
2,fedpairwise_T0.01_d0 vs fedsim_g3,rec,220,0.624716,0.594672,0.030044,0.047273,0.023763,0.036326,62.668903,219,0.0,0.635547,167/220,0.0,Ya,Ya,variant > fedavg
3,fedpairwise_T0.01_d0 vs fedsim_g3,f1,220,0.464396,0.443708,0.020688,0.025192,0.017340,0.024035,80.975771,219,0.0,0.821203,179/220,0.0,Ya,Ya,variant > fedavg
4,fedpairwise_T0.01_d0 vs fedsim_g3,auc,220,0.739817,0.716065,0.023753,0.016017,0.021625,0.025881,146.228972,219,0.0,1.482958,209/220,0.0,Ya,Ya,variant > fedavg
5,fedpairwise_T0.01_d0 vs fedsim_g3,prauc,220,0.439828,0.405765,0.034063,0.027521,0.030406,0.037719,122.045419,219,0.0,1.237704,192/220,0.0,Ya,Ya,variant > fedavg
6,fedpairwise_T0.01_d0.3 vs fedsim_g3,acc,220,0.703572,0.693837,0.009734,0.015411,0.007687,0.011782,62.285454,219,0.0,0.631658,162/220,0.0,Ya,Ya,variant > fedavg
7,fedpairwise_T0.01_d0.3 vs fedsim_g3,prec,220,0.371446,0.355665,0.015781,0.019586,0.013178,0.018383,79.448398,219,0.0,0.805713,180/220,0.0,Ya,Ya,variant > fedavg
8,fedpairwise_T0.01_d0.3 vs fedsim_g3,rec,220,0.627942,0.594672,0.033270,0.048236,0.026860,0.039679,68.011321,219,0.0,0.689726,166/220,0.0,Ya,Ya,variant > fedavg
9,fedpairwise_T0.01_d0.3 vs fedsim_g3,f1,220,0.465513,0.443708,0.021805,0.025821,0.018374,0.025236,83.269829,219,0.0,0.844468,179/220,0.0,Ya,Ya,variant > fedavg


In [8]:
import numpy as np
import pandas as pd
from scipy import stats

# --- data: hasil rata-rata per (algoritma, seed) ---
df = pd.read_csv(
     r"C:\Users\s2\Documents\FIXRKI\FIXbanget\newscema\Full_Client\test_per_client_per_seed.csv"
)

metrik   = ["acc", "prec", "rec", "f1", "auc", "prauc"]
baseline = "fedsim_g5"
varian   = ["fedpairwise_T0.01_d0",
            "fedpairwise_T0.01_d0.3",
            "fedpairwise_T0.01_d0.5"]
ALPHA = 0.05

base = df[df.algoritma == baseline].set_index("seed")

rows = []
for v in varian:
    cur = df[df.algoritma == v].set_index("seed")
    seeds = cur.index.intersection(base.index).sort_values()   # pasangkan per seed

    for m in metrik:
        x = cur.loc[seeds, m].astype(float)
        y = base.loc[seeds, m].astype(float)
        d = (x - y).dropna()
        n = len(d)

        mean_d = d.mean()
        sd_d   = d.std(ddof=1) if n > 1 else np.nan

        if n >= 2 and sd_d > 0:
            t, p = stats.ttest_rel(x.loc[d.index], y.loc[d.index])  # dua sisi
            se   = sd_d / np.sqrt(n)
            tcr  = stats.t.ppf(1 - ALPHA / 2, n - 1)
            ci_lo, ci_hi = mean_d - tcr * se, mean_d + tcr * se
            dz   = mean_d / sd_d                                    # Cohen's dz
        else:
            t = p = ci_lo = ci_hi = dz = np.nan

        rows.append({
            "pembanding": f"{v} vs {baseline}",
            "metrik": m,
            "n": n,
            "mean_variant": x.mean(),
            "mean_fedavg": y.mean(),
            "selisih_mean": mean_d,
            "std_selisih": sd_d,
            "CI95_low": ci_lo,
            "CI95_high": ci_hi,
            "t": t,
            "df": n - 1,
            "p_value": p,
            "cohen_dz": dz,
            "menang": f"{int((d > 0).sum())}/{n}",
        })

hasil4 = pd.DataFrame(rows)

# --- koreksi Holm (untuk 18 uji sekaligus) ---
def holm(pvals):
    p = np.asarray(pvals, dtype=float)
    adj = np.full_like(p, np.nan)
    ok = ~np.isnan(p)
    idx = np.argsort(p[ok])
    pv = p[ok][idx]
    m = len(pv)
    h = np.maximum.accumulate((m - np.arange(m)) * pv)
    h = np.minimum(h, 1.0)
    tmp = np.empty(m)
    tmp[idx] = h
    adj[ok] = tmp
    return adj

hasil4["p_holm"] = holm(hasil4["p_value"])
hasil4["signifikan (p<0.05)"] = np.where(hasil4["p_value"] < ALPHA, "Ya", "Tidak")
hasil4["signifikan (Holm)"]   = np.where(hasil4["p_holm"] < ALPHA, "Ya", "Tidak")
hasil4["arah"] = np.where(hasil4["selisih_mean"] > 0, "variant > fedavg",
                         np.where(hasil4["selisih_mean"] < 0, "variant < fedavg", "sama"))

pd.set_option("display.width", 250, "display.max_columns", None)
print(hasil4.round(4).to_string(index=False))

# hasil.round(6).to_csv(
#     r"C:\Users\s2\Documents\FIXRKI\FIXbanget\newscema\Top20\uji_t_berpasangan.csv",
#     index=False,
# )

                         pembanding metrik   n  mean_variant  mean_fedavg  selisih_mean  std_selisih  CI95_low  CI95_high        t  df  p_value  cohen_dz  menang  p_holm signifikan (p<0.05) signifikan (Holm)             arah
  fedpairwise_T0.01_d0 vs fedsim_g5    acc 220        0.7039       0.6972        0.0067       0.0153    0.0047     0.0087  43.3317 219      0.0    0.4394 150/220     0.0                  Ya                Ya variant > fedavg
  fedpairwise_T0.01_d0 vs fedsim_g5   prec 220        0.3712       0.3611        0.0101       0.0181    0.0077     0.0125  54.8970 219      0.0    0.5567 165/220     0.0                  Ya                Ya variant > fedavg
  fedpairwise_T0.01_d0 vs fedsim_g5    rec 220        0.6247       0.6048        0.0200       0.0419    0.0144     0.0255  46.9480 219      0.0    0.4761 151/220     0.0                  Ya                Ya variant > fedavg
  fedpairwise_T0.01_d0 vs fedsim_g5     f1 220        0.4644       0.4509        0.0135       0.0225

In [9]:
hasil4

,pembanding,metrik,n,mean_variant,mean_fedavg,selisih_mean,std_selisih,CI95_low,CI95_high,t,df,p_value,cohen_dz,menang,p_holm,signifikan (p<0.05),signifikan (Holm),arah
0,fedpairwise_T0.01_d0 vs fedsim_g5,acc,220,0.703870,0.697161,0.006709,0.015266,0.004680,0.008737,43.331691,219,0.0,0.439442,150/220,0.0,Ya,Ya,variant > fedavg
1,fedpairwise_T0.01_d0 vs fedsim_g5,prec,220,0.371214,0.361110,0.010104,0.018150,0.007693,0.012516,54.897009,219,0.0,0.556729,165/220,0.0,Ya,Ya,variant > fedavg
2,fedpairwise_T0.01_d0 vs fedsim_g5,rec,220,0.624716,0.604766,0.019950,0.041902,0.014383,0.025518,46.948023,219,0.0,0.476116,151/220,0.0,Ya,Ya,variant > fedavg
3,fedpairwise_T0.01_d0 vs fedsim_g5,f1,220,0.464396,0.450911,0.013485,0.022511,0.010494,0.016476,59.070320,219,0.0,0.599052,165/220,0.0,Ya,Ya,variant > fedavg
4,fedpairwise_T0.01_d0 vs fedsim_g5,auc,220,0.739817,0.722880,0.016937,0.011911,0.015355,0.018520,140.213316,219,0.0,1.421951,204/220,0.0,Ya,Ya,variant > fedavg
5,fedpairwise_T0.01_d0 vs fedsim_g5,prauc,220,0.439828,0.415598,0.024230,0.023842,0.021062,0.027398,100.210413,219,0.0,1.016268,188/220,0.0,Ya,Ya,variant > fedavg
6,fedpairwise_T0.01_d0.3 vs fedsim_g5,acc,220,0.703572,0.697161,0.006410,0.015343,0.004372,0.008449,41.199013,219,0.0,0.417813,147/220,0.0,Ya,Ya,variant > fedavg
7,fedpairwise_T0.01_d0.3 vs fedsim_g5,prec,220,0.371446,0.361110,0.010336,0.018521,0.007875,0.012797,55.026960,219,0.0,0.558047,165/220,0.0,Ya,Ya,variant > fedavg
8,fedpairwise_T0.01_d0.3 vs fedsim_g5,rec,220,0.627942,0.604766,0.023176,0.041549,0.017655,0.028697,55.002724,219,0.0,0.557801,156/220,0.0,Ya,Ya,variant > fedavg
9,fedpairwise_T0.01_d0.3 vs fedsim_g5,f1,220,0.465513,0.450911,0.014603,0.023043,0.011541,0.017665,62.487785,219,0.0,0.633710,169/220,0.0,Ya,Ya,variant > fedavg
